# 01. Данные и единая предобработка
Синтетика содержит повторяющиеся режимы, GARCH-инновации, чистый эталон и загрязнения. Все этапы используют один DataConfig; нормализатор обучается только на train.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)


In [ ]:
from tsprep.config import DataConfig
from tsprep.data_preprocessing.synthetic import generate_series
from tsprep.data_preprocessing.datasets import prepare_and_window, forecast_pairs
from tsprep.data_preprocessing.preprocessing import inverse_series
series = generate_series(length=3000, seed=2026)
cfg = DataConfig(window_len=48, stride=2, log_transform=True)
prepared = prepare_and_window(series.values, cfg, regime=series.regime,
                              is_anomaly=series.is_anomaly, feature_names=["synthetic_price"])
print({s: getattr(prepared, s + "_windows").shape for s in ("train", "val", "test")})
print("train normalizer:", prepared.normalizer.state())
print("target: transformed observation at exact t +", cfg.forecast_horizon)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
axes[0].plot(series.clean_values[:, 0], label="clean")
axes[0].plot(series.values[:, 0], label="observed", alpha=.6)
axes[0].legend()
axes[1].plot(prepared.raw_normalized[:, 0])
axes[1].set_title("Train-normalized log differences")
axes[2].plot(series.regime)
axes[2].set_title("Recurring regimes")
fig.tight_layout()
restored = inverse_series(prepared.raw_normalized, prepared.normalizer, prepared.y0,
                          cfg.seasonal_period, cfg.log_transform)
np.testing.assert_allclose(restored, series.values)


## Проверка реальных CSV
Подозрительные OHLC-строки не исправляются автоматически. Признаки выбираются явно; нулевой Volume не подходит для логарифмирования.


In [ ]:
from tsprep.data_preprocessing.quality import csv_quality_report
reports = {p.name: csv_quality_report(p) for p in sorted((ROOT / "data" / "raw").glob("*.csv"))}
reports


## Разметка для downstream и парные clean/noisy окна
Forecast target строится по точному индексу end+H. Для разностей anomaly mask учитывает обе исходные точки. Ниже видны поддержка задач и частоты загрязнения до обучения моделей; метки и clean не подмешиваются к входным признакам.

In [ ]:
import pandas as pd
from tsprep.evaluation.downstream import window_anomaly_labels
from tsprep.data_preprocessing.datasets import build_dataloaders
clean_prepared = prepare_and_window(series.clean_values, cfg, normalizer=prepared.normalizer)
support_rows = []
for split in ("train", "val", "test"):
    lo, hi = prepared.split_bounds[split]
    windows = getattr(prepared, split+"_windows")
    ends = prepared.window_starts[split]+cfg.window_len-1
    window_labels = window_anomaly_labels(prepared.is_anomaly[lo:hi], cfg.window_len, cfg.stride, len(windows))
    support_rows.append(dict(split=split, windows=len(windows), forecast_pairs=len(forecast_pairs(prepared, split, cfg)[0]),
        regime_counts=dict(zip(*np.unique(prepared.regime[ends], return_counts=True))),
        anomaly_endpoints=int(prepared.is_anomaly[ends].sum()), contaminated_window_rate=float(window_labels.mean())))
print(pd.DataFrame(support_rows).to_string(index=False))
paired_train, paired_val, _ = build_dataloaders(prepared, cfg, train_targets=clean_prepared.train_windows,
                                              val_targets=clean_prepared.val_windows)
noisy_batch, clean_batch = next(iter(paired_train))
assert noisy_batch.shape == clean_batch.shape
assert clean_prepared.normalizer is prepared.normalizer
print("paired input / target:", noisy_batch.shape, clean_batch.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
n = min(300, len(prepared.raw_normalized))
ax.plot(prepared.raw_normalized[:n, 0], label="observed", alpha=.6)
ax.plot(clean_prepared.raw_normalized[:n, 0], label="clean", linewidth=1)
points = np.flatnonzero(prepared.is_anomaly[:n])
ax.scatter(points, prepared.raw_normalized[points, 0], color="red", s=20, label="contaminated differences")
ax.set_title("Same train-fitted normalization for observed inputs and clean targets")
ax.legend()
plt.show()